In [1]:
# Celda 1: Instalación de librerías
!pip install langchain langgraph langchain-groq -q
print("Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
Librerías instaladas correctamente


In [2]:
# Celda 2: Configuración de API Key
import os
from google.colab import userdata

# Configuramos la API Key de Groq
os.environ["GROQ_API_KEY"] = "clave de github"

print("API Key configurada correctamente")

API Key configurada correctamente


In [3]:
# Celda 3: Conexión con Groq y prueba del modelo
from langchain_groq import ChatGroq

# Usamos gpt-oss-20b disponible en tu cuenta
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# Prueba simple
respuesta = llm.invoke("Hola, ¿puedes responder en español?")
print("Conexión exitosa con Groq")
print(f"Respuesta del modelo: {respuesta.content}")

Conexión exitosa con Groq
Respuesta del modelo: ¡Claro! Puedo responder en español. ¿En qué puedo ayudarte hoy?


In [4]:
# Celda 4: Agente ReAct para consultas bancarias
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent

# --- HERRAMIENTAS DEL AGENTE ---

@tool
def consultar_politica_bancaria(categoria: str) -> str:
    """Consulta la política bancaria según la categoría de la consulta."""
    politicas = {
        "bloqueo_tarjeta": "Para bloquear una tarjeta: verificar identidad con cédula + OTP. El bloqueo es inmediato. Se genera número de caso. Reposición en 3-5 días hábiles.",
        "desbloqueo_tarjeta": "Para desbloquear: verificar que el bloqueo fue voluntario. Si fue por intentos fallidos de PIN, derivar a sucursal.",
        "credito_consumo": "Requisitos crédito consumo: cédula, 3 roles de pago, planilla servicios básicos, score mínimo 650. Tasa 16.5% anual. Cuota máxima 40% del ingreso.",
        "certificado_cuenta": "Certificado de cuenta activa: costo $3.00, emisión inmediata en app/web o 1 hora en sucursal. Incluye saldo promedio 6 meses.",
        "reseteo_clave": "Reseteo clave banca web: opción 'Olvidé mi clave' en login, código OTP al email o teléfono registrado. Nueva clave mínimo 8 caracteres.",
        "desgravamen": "Seguro desgravamen: se da de baja automáticamente al cancelar el crédito. Si hay primas no devengadas, se reembolsan en 5-7 días hábiles.",
    }
    return politicas.get(categoria, "Categoría no encontrada. Categorías disponibles: " + ", ".join(politicas.keys()))

@tool
def verificar_identidad(cedula: str) -> str:
    """Verifica la identidad del cliente con su número de cédula."""
    # Simulación — en producción consultaría la base de datos real
    if len(cedula) == 10 and cedula.isdigit():
        return f"Identidad verificada para cédula {cedula[:3]}*******. Cliente activo."
    return "❌ Número de cédula inválido. Debe tener 10 dígitos."

@tool
def generar_numero_caso(tipo: str) -> str:
    """Genera un número de caso para el seguimiento de la solicitud."""
    import random
    codigo = random.randint(1000, 9999)
    prefijos = {"bloqueo": "BC", "fraude": "FR", "credito": "CR", "certificado": "CERT", "clave": "CL"}
    prefijo = prefijos.get(tipo.lower(), "GEN")
    return f"Número de caso generado: #{prefijo}-2026-{codigo}"

# --- CREAR EL AGENTE REACT ---
herramientas = [consultar_politica_bancaria, verificar_identidad, generar_numero_caso]

agente = create_react_agent(llm, herramientas)

print("Agente ReAct creado con 3 herramientas bancarias")
print("Herramientas disponibles:", [h.name for h in herramientas])

Agente ReAct creado con 3 herramientas bancarias
Herramientas disponibles: ['consultar_politica_bancaria', 'verificar_identidad', 'generar_numero_caso']


/tmp/ipykernel_3878/3522031193.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(llm, herramientas)


In [6]:
# Celda 6: Sistema Baseline (reglas fijas)
# Este es el sistema "tradicional" contra el que comparamos el agente ReAct

def sistema_baseline(consulta: str) -> dict:
    """
    Sistema baseline basado en reglas fijas.
    No razona, no usa herramientas, solo busca palabras clave
    y devuelve respuestas predefinidas.
    """
    consulta_lower = consulta.lower()
    pasos = 1  # El baseline siempre usa 1 solo paso

    # Reglas fijas por palabra clave
    if any(palabra in consulta_lower for palabra in ["bloquear", "bloqueo", "perdí", "perdi", "robaron", "robo"]):
        respuesta = "Para bloquear su tarjeta llame al 1800-BANCO o acérquese a una sucursal con su cédula."
        resuelto = True

    elif any(palabra in consulta_lower for palabra in ["desbloquear", "desbloqueo"]):
        respuesta = "Para desbloquear su tarjeta acérquese a una sucursal con su cédula de identidad."
        resuelto = True

    elif any(palabra in consulta_lower for palabra in ["crédito", "credito", "préstamo", "prestamo", "requisitos"]):
        respuesta = "Para solicitar un crédito necesita: cédula, roles de pago y planilla de servicios básicos. Acérquese a una sucursal."
        resuelto = True

    elif any(palabra in consulta_lower for palabra in ["certificado", "cuenta activa", "no adeudar"]):
        respuesta = "Los certificados se emiten en sucursal con su cédula. El costo es de $3.00 y se entrega en el momento."
        resuelto = True

    elif any(palabra in consulta_lower for palabra in ["clave", "contraseña", "pin", "bloqueado"]):
        respuesta = "Para resetear su clave acérquese a una sucursal con su cédula o llame al 1800-BANCO."
        resuelto = True

    elif any(palabra in consulta_lower for palabra in ["desgravamen", "seguro", "cancelar seguro"]):
        respuesta = "Para cancelar el seguro de desgravamen acérquese a una sucursal con su cédula y tabla de pagos."
        resuelto = True

    elif any(palabra in consulta_lower for palabra in ["actualizar", "actualización", "datos", "dirección", "telefono"]):
        respuesta = "Para actualizar sus datos acérquese a una sucursal con su cédula de identidad."
        resuelto = True

    else:
        respuesta = "Para más información comuníquese con nuestro centro de atención al cliente al 1800-BANCO."
        resuelto = False  # No pudo resolver la consulta

    return {
        "respuesta": respuesta,
        "pasos": pasos,
        "resuelto": resuelto
    }

# Prueba del baseline con el mismo escenario E1.1
consulta_prueba = "Necesito bloquear mi tarjeta de débito, la perdí esta mañana en el transporte."

resultado_baseline = sistema_baseline(consulta_prueba)

print("="*60)
print("SISTEMA BASELINE — Prueba con escenario E1.1")
print("="*60)
print(f"Consulta: {consulta_prueba}")
print(f"\nRespuesta: {resultado_baseline['respuesta']}")
print(f"Pasos usados: {resultado_baseline['pasos']}")
print(f"¿Resuelto correctamente?: {resultado_baseline['resuelto']}")
print("="*60)

SISTEMA BASELINE — Prueba con escenario E1.1
Consulta: Necesito bloquear mi tarjeta de débito, la perdí esta mañana en el transporte.

Respuesta: Para bloquear su tarjeta llame al 1800-BANCO o acérquese a una sucursal con su cédula.
Pasos usados: 1
¿Resuelto correctamente?: True


In [7]:
# Celda 7: Experimento completo — 18 escenarios
import time
import json

# Los 18 escenarios de prueba
escenarios_prueba = [
    # CATEGORÍA 1: Bloqueo/Desbloqueo
    {"id": "E1.1", "categoria": "Bloqueo tarjeta", "complejidad": "Baja",
     "consulta": "Necesito bloquear mi tarjeta de débito, la perdí esta mañana en el transporte. Mi cédula es 1723456789.",
     "elementos_esperados": ["bloqueo", "caso", "reposición"]},
    {"id": "E1.2", "categoria": "Desbloqueo tarjeta", "complejidad": "Media",
     "consulta": "Quiero desbloquear mi tarjeta de crédito, la bloqueé por error desde la app. Cédula 0912345678.",
     "elementos_esperados": ["desbloqueo", "verificar", "pin"]},
    {"id": "E1.3", "categoria": "Bloqueo por fraude", "complejidad": "Alta",
     "consulta": "Mi tarjeta fue robada ayer y hicieron compras no autorizadas por $847. Cédula 1756789012.",
     "elementos_esperados": ["bloqueo", "fraude", "reclamación", "caso"]},

    # CATEGORÍA 2: Crédito
    {"id": "E2.1", "categoria": "Requisitos crédito", "complejidad": "Baja",
     "consulta": "¿Cuáles son los requisitos para solicitar un crédito de consumo de $5.000?",
     "elementos_esperados": ["cédula", "requisitos", "tasa"]},
    {"id": "E2.2", "categoria": "Estado solicitud", "complejidad": "Media",
     "consulta": "¿En qué estado está mi solicitud de crédito hipotecario #SOL-2026-1547?",
     "elementos_esperados": ["estado", "etapa", "tiempo"]},
    {"id": "E2.3", "categoria": "Capacidad endeudamiento", "complejidad": "Alta",
     "consulta": "Tengo ingresos de $2.000, crédito vigente de $8.000 con cuota de $198. ¿Puedo acceder a $3.000 adicionales?",
     "elementos_esperados": ["capacidad", "40%", "cuota", "califica"]},

    # CATEGORÍA 3: Actualización datos
    {"id": "E3.1", "categoria": "Actualizar contacto", "complejidad": "Baja",
     "consulta": "Necesito actualizar mi número de teléfono y dirección en el sistema del banco.",
     "elementos_esperados": ["OTP", "app", "sucursal"]},
    {"id": "E3.2", "categoria": "Actualizar empresa", "complejidad": "Media",
     "consulta": "Quiero actualizar los datos de mi empresa porque cambié de razón social.",
     "elementos_esperados": ["RUC", "documentos", "sucursal", "días"]},
    {"id": "E3.3", "categoria": "Cuenta bloqueada", "complejidad": "Alta",
     "consulta": "Mi empleado no puede actualizar sus datos bancarios porque tiene la cuenta bloqueada por inactividad.",
     "elementos_esperados": ["inactividad", "reactivación", "RRHH", "plazo"]},

    # CATEGORÍA 4: Certificados
    {"id": "E4.1", "categoria": "Certificado cuenta", "complejidad": "Baja",
     "consulta": "Necesito un certificado de cuenta activa para presentar al consulado.",
     "elementos_esperados": ["costo", "emisión", "app", "sucursal"]},
    {"id": "E4.2", "categoria": "Certificado no adeudar", "complejidad": "Media",
     "consulta": "Necesito un certificado de que no tengo deudas vencidas para un concurso público.",
     "elementos_esperados": ["vencidas", "verificación", "costo", "tiempo"]},
    {"id": "E4.3", "categoria": "Certificado retenciones", "complejidad": "Alta",
     "consulta": "Soy del área contable y necesito certificados de retenciones 2025 para 5 empleados. El SRI vence en 10 días.",
     "elementos_esperados": ["portal", "masiva", "días hábiles", "SRI"]},

    # CATEGORÍA 5: Claves
    {"id": "E5.1", "categoria": "Reseteo clave web", "complejidad": "Baja",
     "consulta": "Olvidé mi clave de la banca web, ¿cómo la recupero?",
     "elementos_esperados": ["OTP", "email", "nueva clave"]},
    {"id": "E5.2", "categoria": "PIN bloqueado", "complejidad": "Media",
     "consulta": "Mi PIN de cajero está bloqueado después de tres intentos fallidos. ¿Qué hago?",
     "elementos_esperados": ["app", "cajero", "sucursal", "NFC"]},
    {"id": "E5.3", "categoria": "Clave sistema interno", "complejidad": "Alta",
     "consulta": "Soy jefe de agencia y necesito resetear la clave del sistema interno de un operador que lleva 2 horas sin poder trabajar.",
     "elementos_esperados": ["mesa de ayuda", "ticket", "autoridad", "urgente"]},

    # CATEGORÍA 6: Desgravamen
    {"id": "E6.1", "categoria": "Cancelar desgravamen", "complejidad": "Media",
     "consulta": "¿Cómo cancelo el seguro de desgravamen de mi crédito que ya pagué en su totalidad?",
     "elementos_esperados": ["automática", "reembolso", "primas"]},
    {"id": "E6.2", "categoria": "Seguro no solicitado", "complejidad": "Alta",
     "consulta": "Me están cobrando $18.50 mensual por un seguro de desgravamen que nunca pedí, desde hace 8 meses.",
     "elementos_esperados": ["reclamación", "reembolso", "148", "legal"]},
    {"id": "E6.3", "categoria": "Siniestro fallecimiento", "complejidad": "Alta",
     "consulta": "Un cliente falleció y tiene crédito vigente con saldo $4.200 y seguro de desgravamen activo. ¿Cómo aplicamos el seguro?",
     "elementos_esperados": ["defunción", "documentos", "días hábiles", "intereses"]},
]

def evaluar_respuesta(respuesta: str, elementos_esperados: list) -> float:
    """Evalúa qué porcentaje de elementos esperados contiene la respuesta."""
    respuesta_lower = respuesta.lower()
    encontrados = sum(1 for elem in elementos_esperados if elem.lower() in respuesta_lower)
    return round(encontrados / len(elementos_esperados), 2)

# EJECUTAR EXPERIMENTO
print("🔬 INICIANDO EXPERIMENTO COMPARATIVO")
print("="*60)
print(f"Total escenarios: {len(escenarios_prueba)}")
print(f"Sistemas a comparar: Agente ReAct vs. Baseline")
print("="*60)

resultados = []

for i, escenario in enumerate(escenarios_prueba):
    print(f"\n📋 Escenario {escenario['id']} — {escenario['categoria']} ({escenario['complejidad']})")

    # 1. BASELINE
    inicio = time.time()
    res_baseline = sistema_baseline(escenario["consulta"])
    tiempo_baseline = round(time.time() - inicio, 3)
    precision_baseline = evaluar_respuesta(res_baseline["respuesta"], escenario["elementos_esperados"])

    # 2. AGENTE ReAct
    inicio = time.time()
    try:
        res_react = agente.invoke({
            "messages": [
                {"role": "system", "content": "Eres un agente de soporte bancario. Resuelve la consulta usando las herramientas disponibles. Responde en español."},
                {"role": "user", "content": escenario["consulta"]}
            ]
        })
        respuesta_react = res_react["messages"][-1].content
        pasos_react = sum(1 for m in res_react["messages"] if "ToolMessage" in type(m).__name__)
        resuelto_react = True
    except Exception as e:
        respuesta_react = f"Error: {str(e)}"
        pasos_react = 0
        resuelto_react = False

    tiempo_react = round(time.time() - inicio, 2)
    precision_react = evaluar_respuesta(respuesta_react, escenario["elementos_esperados"])

    # GUARDAR RESULTADO
    resultado = {
        "id": escenario["id"],
        "categoria": escenario["categoria"],
        "complejidad": escenario["complejidad"],
        "baseline": {
            "resuelto": res_baseline["resuelto"],
            "precision": precision_baseline,
            "pasos": res_baseline["pasos"],
            "tiempo": tiempo_baseline
        },
        "react": {
            "resuelto": resuelto_react,
            "precision": precision_react,
            "pasos": pasos_react,
            "tiempo": tiempo_react
        }
    }
    resultados.append(resultado)

    print(f"   Baseline → Precisión: {precision_baseline:.0%} | Pasos: {res_baseline['pasos']} | Tiempo: {tiempo_baseline}s")
    print(f"   ReAct    → Precisión: {precision_react:.0%} | Pasos: {pasos_react} | Tiempo: {tiempo_react}s")

print("\n EXPERIMENTO COMPLETADO")
print(f"Total escenarios ejecutados: {len(resultados)}")

🔬 INICIANDO EXPERIMENTO COMPARATIVO
Total escenarios: 18
Sistemas a comparar: Agente ReAct vs. Baseline

📋 Escenario E1.1 — Bloqueo tarjeta (Baja)
   Baseline → Precisión: 0% | Pasos: 1 | Tiempo: 0.0s
   ReAct    → Precisión: 100% | Pasos: 4 | Tiempo: 3.05s

📋 Escenario E1.2 — Desbloqueo tarjeta (Media)
   Baseline → Precisión: 0% | Pasos: 1 | Tiempo: 0.0s
   ReAct    → Precisión: 67% | Pasos: 4 | Tiempo: 2.98s

📋 Escenario E1.3 — Bloqueo por fraude (Alta)
   Baseline → Precisión: 0% | Pasos: 1 | Tiempo: 0.0s
   ReAct    → Precisión: 75% | Pasos: 4 | Tiempo: 2.46s

📋 Escenario E2.1 — Requisitos crédito (Baja)
   Baseline → Precisión: 33% | Pasos: 1 | Tiempo: 0.0s
   ReAct    → Precisión: 100% | Pasos: 2 | Tiempo: 2.12s

📋 Escenario E2.2 — Estado solicitud (Media)
   Baseline → Precisión: 0% | Pasos: 1 | Tiempo: 0.0s
   ReAct    → Precisión: 33% | Pasos: 0 | Tiempo: 3.8s

📋 Escenario E2.3 — Capacidad endeudamiento (Alta)
   Baseline → Precisión: 0% | Pasos: 1 | Tiempo: 0.0s
   ReAct    

In [9]:
# Celda 8: Análisis de resultados y métricas finales
import statistics

# Recopilar métricas
precisiones_baseline = [r["baseline"]["precision"] for r in resultados]
precisiones_react = [r["react"]["precision"] for r in resultados]
pasos_react = [r["react"]["pasos"] for r in resultados]
tiempos_react = [r["react"]["tiempo"] for r in resultados]

resueltos_baseline = sum(1 for r in resultados if r["baseline"]["resuelto"])
resueltos_react = sum(1 for r in resultados if r["react"]["resuelto"])

print("="*60)
print(" RESULTADOS DEL EXPERIMENTO COMPARATIVO")
print("="*60)

print("\n SISTEMA BASELINE (Reglas fijas):")
print(f"   Tasa de resolución: {resueltos_baseline}/{len(resultados)} ({resueltos_baseline/len(resultados):.0%})")
print(f"   Precisión promedio: {statistics.mean(precisiones_baseline):.0%}")
print(f"   Pasos por consulta: 1 (siempre)")
print(f"   Tiempo promedio: ~0.001s")

print("\ AGENTE ReAct:")
print(f"   Tasa de resolución: {resueltos_react}/{len(resultados)} ({resueltos_react/len(resultados):.0%})")
print(f"   Precisión promedio: {statistics.mean(precisiones_react):.0%}")
print(f"   Pasos promedio: {statistics.mean(pasos_react):.1f}")
print(f"   Tiempo promedio: {statistics.mean(tiempos_react):.1f}s")

print("\n COMPARATIVA:")
mejora_precision = statistics.mean(precisiones_react) - statistics.mean(precisiones_baseline)
print(f"   Mejora en precisión: +{mejora_precision:.0%}")
print(f"   Pasos de razonamiento adicionales: +{statistics.mean(pasos_react):.1f}")

print("\n DETALLE POR COMPLEJIDAD:")
for complejidad in ["Baja", "Media", "Alta"]:
    grupo = [r for r in resultados if r["complejidad"] == complejidad]
    prec_b = statistics.mean([r["baseline"]["precision"] for r in grupo])
    prec_r = statistics.mean([r["react"]["precision"] for r in grupo])
    print(f"   {complejidad}: Baseline {prec_b:.0%} vs ReAct {prec_r:.0%}")

print("="*60)
print("✅ Análisis completado — Datos listos para el Capítulo 4")

 RESULTADOS DEL EXPERIMENTO COMPARATIVO

 SISTEMA BASELINE (Reglas fijas):
   Tasa de resolución: 17/18 (94%)
   Precisión promedio: 11%
   Pasos por consulta: 1 (siempre)
   Tiempo promedio: ~0.001s
\ AGENTE ReAct:
   Tasa de resolución: 18/18 (100%)
   Precisión promedio: 34%
   Pasos promedio: 1.9
   Tiempo promedio: 8.2s

 COMPARATIVA:
   Mejora en precisión: +23%
   Pasos de razonamiento adicionales: +1.9

 DETALLE POR COMPLEJIDAD:
   Baja: Baseline 23% vs ReAct 47%
   Media: Baseline 12% vs ReAct 29%
   Alta: Baseline 0% vs ReAct 29%
✅ Análisis completado — Datos listos para el Capítulo 4


<>:23: SyntaxWarning: invalid escape sequence '\ '
<>:23: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_3878/2965293574.py:23: SyntaxWarning: invalid escape sequence '\ '
  print("\ AGENTE ReAct:")
